# AQNet — Three-Tier PM2.5 Research Pipeline

Runs the **AQNet** offline research model of the [Shared Skies Initiative](https://sharedskiesinitiative.org/real-time-map)
end to end on a fresh Colab runtime. AQNet is a research track — it does **not** feed the live map.

**The three tiers**

1. **Tier 1 — tabular:** GBM ensemble (LightGBM / XGBoost / CatBoost / Random Forest) with
   leave-one-sensor-out (LOSO) cross-validation, per-fold-recomputed neighbor features, and LightGBM
   quantile heads. The headline blend is **leave-one-fold-out**: simplex weights are fit without the
   scored fold's rows. XGBoost/CatBoost GPU modes enable automatically when CUDA is present.
2. **Tier 2 — deep:** the existing `FusionUNet` (per-source spatial-attention fusion → U-Net) on a gridded
   0.1° Texas stack extended with GEOS-CF / MERRA-2 channels, a CAMS dust plane, day-of-week encodings,
   and per-source availability **flag** channels — trained with AMP, warmup + cosine LR, masked Huber
   loss, random crops, and modality dropout.
3. **Tier 3 — fusion:** residual kriging of Tier-1 errors + a cross-fit ridge meta-learner over strictly
   out-of-fold parts + conformalized-quantile (CQR-style) prediction intervals.

Validation: LOSO, spatial-block, and temporal CV; interpolation and raw / mean-debiased CTM baselines;
cluster (per-sensor) bootstrap CIs; per-day residual Moran's I; smoke / dust / clean strata metrics;
a spatial vs temporal R² decomposition; a feature-set **ablation** stage; and **external EPA AQS
FRM/FEM monitors that the models never train on**, including a deployment-mode Tier-3 row.

**Before you start:** switch to a GPU runtime (`Runtime → Change runtime type → GPU`). A hard GPU
check below stops the notebook before the full run if CUDA is missing — the tabular, ablation, and
validate stages are the CPU long poles, but the deep stage needs a GPU to finish in about an hour
instead of days.

No performance numbers are promised anywhere in this notebook — every metric you see at the end is
computed by the run itself.

In [ ]:
# 1) Clone the repo (~400 MB of data included — this takes a minute or two)
import os

if os.path.basename(os.getcwd()) != "shared-skies-initiative":
    if not os.path.isdir("shared-skies-initiative"):
        print("Cloning shared-skies-initiative (~400 MB)...")
        !git clone --depth 1 https://github.com/saketh-c/shared-skies-initiative
    %cd shared-skies-initiative
print("Working directory:", os.getcwd())

## 2) Optional: mount Google Drive (survive session loss)

Colab runtimes are ephemeral — a disconnect wipes the local disk, including multi-hour MERRA-2
downloads and trained checkpoints. This cell mounts your Google Drive and symlinks
`research/aqnet/artifacts/`, `cache/`, and `data/` into `MyDrive/aqnet/`, so caches, artifacts, and
checkpoints all survive a lost session. Re-running the notebook later picks them straight back up:
the stages are restartable and the external caches are window-aware, so nothing is re-downloaded or
recomputed unnecessarily.

**Safe to skip** — set `MOUNT_DRIVE = False` (or decline the authorization prompt) and everything
then lives on the runtime disk and dies with the session.

In [ ]:
# 2) Optional Drive mount — safe to skip
MOUNT_DRIVE = True  # set False to keep everything on the ephemeral VM disk
import os, shutil

if MOUNT_DRIVE:
    try:
        from google.colab import drive
        drive.mount("/content/drive")
        base = "/content/drive/MyDrive/aqnet"
        for sub in ("artifacts", "cache", "data"):
            target = os.path.join(base, sub)
            link = os.path.join("research/aqnet", sub)
            os.makedirs(target, exist_ok=True)
            if os.path.islink(link):
                print(f"{link} already links into Drive.")
                continue
            if os.path.isdir(link):
                # keep anything already produced locally; Drive copies win ties
                for name in os.listdir(link):
                    dst = os.path.join(target, name)
                    if not os.path.exists(dst):
                        shutil.move(os.path.join(link, name), dst)
                shutil.rmtree(link)
            os.symlink(target, link, target_is_directory=True)
            print(f"{link} -> {target}")
    except Exception as e:
        print(f"Drive mount skipped ({e}) — artifacts stay on the ephemeral "
              "runtime disk.")
else:
    print("MOUNT_DRIVE = False — artifacts stay on the ephemeral runtime disk.")

In [ ]:
# 3) Install AQNet dependencies (torch ships with Colab; the rest is quick)
%pip install -q -r research/aqnet/requirements.txt

## 4) Optional: NASA Earthdata login (enables MERRA-2 aerosol species + PBLH)

MERRA-2 requires a **free** NASA Earthdata account:

1. Register at <https://urs.earthdata.nasa.gov/users/new> (takes ~2 minutes).
2. Run the cell below and enter your username and password when prompted.

MERRA-2 contributes aerosol-species features/channels and daily PBLH mean/max/min features. The
first full-window fetch can take **hours**; it streams month by month into cached chunks (on Drive
if you mounted it above), so you only ever pay that cost once.

**This step is entirely skippable.** If you skip it (or the login fails), the pipeline runs with
`--skip-merra2` automatically: the `merra2_*` feature columns are simply NaN and every downstream
stage degrades gracefully. The quick smoke test below skips MERRA-2 regardless of this login.

In [ ]:
# 4) Optional Earthdata login — safe to skip
MERRA2_FLAG = "--skip-merra2"
try:
    import earthaccess
    auth = earthaccess.login(strategy="interactive", persist=True)
    if getattr(auth, "authenticated", False):
        MERRA2_FLAG = ""
        print("Earthdata login OK — the data stage will fetch MERRA-2.")
    else:
        print("Not authenticated — continuing with --skip-merra2 "
              "(merra2_* features -> NaN).")
except Exception as e:
    print(f"Earthdata login skipped ({e}) — continuing with --skip-merra2.")
print("MERRA2_FLAG =", repr(MERRA2_FLAG))

## 5) Smoke test first (recommended)

`--quick` runs **every** stage on a small slice — a 3-month 2024 window, a coarser 0.2° grid, 4 LOSO
folds, and 3 U-Net epochs — so you can validate the whole plumbing (downloads, feature build, all three
tiers, ablation, validation artifacts) before committing to the multi-hour full run. Expect roughly
5–15 minutes, dominated by the external downloads.

**The quick run always passes `--skip-merra2`, regardless of the login above.** The Earthdata fetch is
by far the slowest external source — bulky hourly granules that would turn a minutes-long plumbing
check into an hours-long download — and `--quick` exists to check plumbing, not to download aerosol
reanalysis; `merra2_*` columns are NaN and every stage degrades gracefully. The full data stage below
still fetches MERRA-2 if you logged in.

Quick-then-full is safe: final external caches embed their date window in the filename and are
validated against the requested window (month chunks are the cache of record), so the full run never
mistakes the quick run's 3-month caches for full-window data, and nothing the quick run downloads is
wasted.

Artifacts land in `research/aqnet/artifacts/` and are **overwritten by the full run below**, so treat
quick outputs as plumbing checks, not results.

In [ ]:
# 5) Quick end-to-end smoke test (~5-15 min, mostly download time)
# --skip-merra2 is forced here on purpose — see the note above.
!python research/aqnet/pipeline_colab.py all --quick --skip-merra2

## 6) GPU gate — required before the full run

The deep stage is impractical on CPU, and AMP, the larger default batch size, and the XGBoost/CatBoost
GPU modes all key off this device. This cell hard-stops the notebook if CUDA is unavailable so you
find out now, not hours in.

In [ ]:
# 6) Hard GPU check — the full run assumes CUDA from here on
import torch

assert torch.cuda.is_available(), (
    "No CUDA GPU detected. In Colab: Runtime -> Change runtime type -> GPU "
    "(a T4 is enough), then re-run from the top. The deep stage would take "
    "days on CPU."
)
print("GPU OK:", torch.cuda.get_device_name(0))

## 7) Full run, stage by stage

Rough expectations on a Colab GPU runtime (network speed and library versions move these a lot):

| Stage | What it does | Rough runtime |
|---|---|---|
| `data` | EPA AQS zips + GEOS-CF OPeNDAP (+ MERRA-2 if logged in), window-stamped caches | minutes without MERRA-2; the first full-window MERRA-2 fetch can take **hours** (month chunks cached — you pay once) |
| `features` | training frame + per-fold neighbor features + frozen CV folds | minutes to tens of minutes |
| `tabular` | Tier-1 LOSO CV (4 models × folds) + LOFO blend + quantile heads | **hours** — a CPU long pole (GPU helps XGBoost/CatBoost) |
| `deep` | extended grid stack + FusionUNet (AMP, crops, flag channels) | ~1–1.5 h on a T4 |
| `fuse` | residual kriging + cross-fit meta-learner + conformal calibration | tens of minutes (per-day kriging) |
| `ablation` | Tier-1 re-run on the frozen folds per feature-set variant | roughly the tabular stage's cost per variant (`--skip-ablation` drops it from `all`) |
| `validate` | spatial/temporal CV retrains, baselines, strata, per-day Moran's I, external AQS + Tier-3-at-AQS, SHAP | **hours** — the other CPU long pole |

Each stage is restartable — if the runtime disconnects, re-run from the stage that failed (with Drive
mounted, completed artifacts survive the disconnect). Stages print what they skipped and why. The deep
stage picks its batch size automatically (32 on GPU); pass `--batch-size N` to override.

In [ ]:
# 7a) data — fetch external open datasets (window-aware caches)
MERRA2_FLAG = globals().get("MERRA2_FLAG", "--skip-merra2")
!python research/aqnet/pipeline_colab.py data {MERRA2_FLAG}

In [ ]:
# 7b) features — training frame + per-fold neighbor features + frozen folds
!python research/aqnet/pipeline_colab.py features

In [ ]:
# 7c) tabular — Tier 1: GBM ensemble LOSO CV + LOFO blend + quantile heads
!python research/aqnet/pipeline_colab.py tabular

In [ ]:
# 7d) deep — Tier 2: FusionUNet on the extended stack (~1-1.5 h on a T4)
# Add --batch-size N to override the automatic batch size (32 on GPU).
!python research/aqnet/pipeline_colab.py deep

In [ ]:
# 7e) fuse — Tier 3: residual kriging + cross-fit meta-learner + conformal
!python research/aqnet/pipeline_colab.py fuse

In [ ]:
# 7f) ablation — Tier-1 feature-set variants on the SAME frozen LOSO folds
# (primary / plus_demographics / no_external / no_neighbor).
# plus_demographics exists ONLY as an ablation — demographic columns are
# never inputs to any reported tier.
!python research/aqnet/pipeline_colab.py ablation

In [ ]:
# 7g) validate — all metrics artifacts + SUMMARY.md
!python research/aqnet/pipeline_colab.py validate

## 8) Results

The cells below only display what the run produced — each `metrics_*.json` as a table, the ablation
deltas, the conformal / strata / Moran blocks, the SHAP summary plot, and the auto-generated
`SUMMARY.md`. Missing files mean the corresponding stage was skipped (the stage logs above say why).

In [ ]:
# 8a) Every metrics_*.json as a table
import glob, json, os
import pandas as pd
from IPython.display import display, Markdown

ART = "research/aqnet/artifacts"
found = sorted(glob.glob(os.path.join(ART, "metrics_*.json")))
if not found:
    print("No metrics_*.json artifacts yet — run the validate stage first.")
for path in found:
    with open(path) as f:
        obj = json.load(f)
    display(Markdown(f"### `{os.path.basename(path)}`"))
    table = pd.json_normalize(obj, sep=".").T
    table.columns = ["value"]
    display(table)

In [ ]:
# 8b) Ablation — per-variant LOSO metrics + paired delta-R2 vs primary
import json, os
import pandas as pd
from IPython.display import display, Markdown

p = "research/aqnet/artifacts/metrics_ablation.json"
if os.path.exists(p):
    with open(p) as f:
        abl = json.load(f)
    display(Markdown("### `metrics_ablation.json` — feature-set ablation "
                     "(paired delta-R2 vs primary, cluster bootstrap CIs)"))
    variant_rows = {k: pd.json_normalize(v, sep=".").iloc[0]
                    for k, v in abl.items() if isinstance(v, dict)}
    if variant_rows:
        display(pd.DataFrame(variant_rows).T)  # rows = variants
    else:
        table = pd.json_normalize(abl, sep=".").T
        table.columns = ["value"]
        display(table)
    display(Markdown(
        "_`plus_demographics` exists **only** as an ablation to quantify the "
        "cost of excluding demographic columns — no reported tier ever uses "
        "them as inputs._"))
else:
    print("metrics_ablation.json not present — run the ablation stage "
          "(or drop --skip-ablation) first.")

In [ ]:
# 8c) Conformal coverage/width, strata, per-day Moran's I, spatial vs temporal R2
import glob, json, os
import pandas as pd
from IPython.display import display, Markdown

ART = "research/aqnet/artifacts"
WANT = ("conformal", "strata", "moran", "spatial_temporal", "spatial_r2")
shown = 0
for path in sorted(glob.glob(os.path.join(ART, "metrics_*.json"))):
    with open(path) as f:
        obj = json.load(f)
    if not isinstance(obj, dict):
        continue
    for key, val in obj.items():
        if not any(w in key.lower() for w in WANT):
            continue
        shown += 1
        display(Markdown(f"### `{os.path.basename(path)}` — `{key}`"))
        block = val if isinstance(val, dict) else {"value": val}
        table = pd.json_normalize(block, sep=".").T
        table.columns = ["value"]
        display(table)
if not shown:
    print("No conformal/strata/Moran blocks found yet — run the fuse and "
          "validate stages first.")

In [ ]:
# 8d) SHAP summary plot (Tier-1 LightGBM)
import os
from IPython.display import Image, display

p = "research/aqnet/artifacts/shap_summary.png"
if os.path.exists(p):
    display(Image(p))
else:
    print("shap_summary.png not present — the interpretability step was "
          "skipped (see the validate-stage log).")

In [ ]:
# 8e) Auto-generated run summary
import os
from IPython.display import Markdown, display

p = "research/aqnet/artifacts/SUMMARY.md"
if os.path.exists(p):
    with open(p) as f:
        display(Markdown(f.read()))
else:
    print("SUMMARY.md not present — run the validate stage first.")